In [16]:
"""
=============================================================================
 01_data_cleaning.py
 Oil Trading Desk Forecasting Project -- Data Preparation Stage
=============================================================================
Purpose
-------
Builds a single "model-ready" monthly dataset (Dec-1994 to Jun-2024) to
forecast next-month CL1 (front-month WTI futures) log returns, with a focus
on testing whether SEASONAL patterns add value, especially in HIGH-VOLATILITY
regimes (expanding VIX threshold).

Steps
-----
1. Load the three monthly sheets (Industry_M, Macro_M, StockPrices_M) and
   align them on a Year-Month key. The model is still estimated at MONTHLY
   frequency (that remains the common denominator across all data sources,
   and matches the seasonal research question -- summer driving / winter
   heating demand are calendar-month phenomena, not daily ones).
1b. HOWEVER, we do not throw away the much larger daily (Industry_D, ~8,700
   rows) and weekly (Industry_W, ~1,800 rows) sheets. Monthly fundamentals
   (inventories, rig counts, macro data) are only ever reported monthly, so
   forcing them onto a daily grid would just forward-fill stale values and
   create fake precision. Instead we use the HIGH-FREQUENCY sheets to build
   additional WITHIN-MONTH engineered features (realised volatility, max
   drawdown, down-day fraction, within-month dispersion of weekly inventory
   and positioning data) that genuinely add new information at the monthly
   level, rather than inflating the row count with duplicated values. This
   uses essentially all rows in the file while keeping one clean frequency.
   NOTE: the daily CL1 series contains the well-known 20-Apr-2020 negative
   WTI print (-$37.63, a one-off storage/expiry anomaly). A standard
   log-return breaks across this (log of a negative number), which would
   silently delete the two most extreme days of the entire 30-year sample
   right when we need them most. We therefore use a sign-robust simple
   return, ret_t = (P_t - P_{t-1}) / |P_{t-1}|, for all within-month
   daily-return features. This correctly registers 20-Apr-2020 as an
   extreme crash and 21-Apr-2020 as a sharp rebound.
2. Drop Brent (redundant benchmark vs. CL1/CL2 WTI contracts; not needed for
   a WTI-specific desk and avoids collinearity). Its daily counterpart is
   still used to build a Brent-WTI dislocation volatility feature (1b).
3. Transform every series to a stationary unit:
      - price-like series      -> log returns
      - quantity/level series  -> log differences (growth rates)
      - interest rates         -> first differences (level changes)
      - spreads (crack spread) -> first differences
      - VIX                    -> kept in level (needed for the regime
                                   threshold) AND as a first-difference
                                   predictor
      - DXY returns            -> already supplied as returns, used as-is
4. Lag every predictor by one month (t-1 information set) to forecast the
   return realised in month t -- i.e. no look-ahead bias.
5. Add seasonal dummies (calendar month, "summer driving season", "winter
   heating season") keyed to the month being forecast (these are known in
   advance, so they are NOT lagged).
6. Define a High-Volatility regime using an EXPANDING (information at time t
   only) percentile of the VIX, with a burn-in window, and build
   seasonal x High-Vol interaction terms.
7. Drop the burn-in rows that cannot be computed (due to differencing and the
   regime burn-in) and export the final model-ready dataset + a data
   dictionary.

Output
------
- model_ready_dataset.xlsx  (sheet 1: Model_Ready_Data, sheet 2: Data_Dictionary)
- model_ready_dataset.csv
=============================================================================
"""

import glob
import os
import numpy as np
import pandas as pd

# -----------------------------------------------------------------------
# 0. CONFIG  (documented, easy-to-defend choices)
# -----------------------------------------------------------------------
# The raw file is called "Oil Data.xlsx" (capital letters + a space) when
# uploaded via the Colab file browser. Rather than hard-coding a single
# path/spelling (which breaks as soon as the filename or folder differs),
# we search a short list of likely locations/spellings and fall back to a
# case-insensitive scan of the current working directory. This makes the
# script portable across Colab, a local machine, or a shared Drive folder.
CANDIDATE_PATHS = [
    "Oil Data.xlsx",
    "oil_data.xlsx",
    "oil data.xlsx",
    "OilData.xlsx",
    "/content/Oil Data.xlsx",          # default Colab upload location
    "/content/drive/MyDrive/Oil Data.xlsx",
]


def find_input_file(candidates):
    """Return the first existing path from `candidates`; otherwise fall
    back to a case-insensitive search for any .xlsx file containing
    'oil' in the current directory (and /content/ if present)."""
    for path in candidates:
        if os.path.exists(path):
            return path

    search_dirs = [".", "/content"]
    for d in search_dirs:
        if not os.path.isdir(d):
            continue
        for f in os.listdir(d):
            if f.lower().endswith(".xlsx") and "oil" in f.lower():
                return os.path.join(d, f)

    raise FileNotFoundError(
        "Could not locate the raw data file. Please upload 'Oil Data.xlsx' "
        "(e.g. via the Colab Files panel, or Files > Upload) and re-run, "
        "or edit CANDIDATE_PATHS / INPUT_FILE below to point at its exact "
        "location."
    )


INPUT_FILE = find_input_file(CANDIDATE_PATHS)
OUTPUT_FILE = "model_ready_dataset.xlsx"
print(f"Using input file: {INPUT_FILE}")

VIX_QUANTILE = 0.70          # "high-volatility" = top 30% historically
VIX_BURNIN = 36              # months of history required before the first
                              # expanding VIX threshold is trusted (3 years)

SUMMER_MONTHS = [5, 6, 7, 8, 9]        # May-Sep: US gasoline driving season
WINTER_MONTHS = [11, 12, 1, 2, 3]      # Nov-Mar: heating oil demand season

# -----------------------------------------------------------------------
# 1. LOAD & ALIGN ON YEAR-MONTH  (monthly sheets: the modelling frequency)
# -----------------------------------------------------------------------
xls = pd.ExcelFile(INPUT_FILE)
ind = pd.read_excel(xls, "Industry_M")
macro = pd.read_excel(xls, "Macro_M").dropna(how="all")
stock = pd.read_excel(xls, "StockPrices_M")

ind.columns = [c.strip() for c in ind.columns]
macro.columns = [c.strip() for c in macro.columns]
stock.columns = [c.strip() for c in stock.columns]

for df_ in (ind, macro, stock):
    df_["ym"] = df_["Date"].dt.to_period("M")

df = (
    ind.merge(macro, on="ym", suffixes=("", "_macro"))
       .merge(stock, on="ym", suffixes=("", "_stock"))
       .sort_values("ym")
       .reset_index(drop=True)
)

# canonical month-end date index
df["Date"] = df["ym"].dt.to_timestamp("M")
df = df.drop(columns=[c for c in df.columns if c.startswith("Date_")])

assert df["ym"].is_unique, "Duplicate year-month after merge!"
print(f"Aligned monthly panel: {df.shape[0]} months, "
      f"{df['Date'].min().date()} to {df['Date'].max().date()}")

# -----------------------------------------------------------------------
# 1b. DAILY & WEEKLY SHEETS -> WITHIN-MONTH ENGINEERED FEATURES
# -----------------------------------------------------------------------
# These sheets have far more rows (~8,700 daily, ~1,800 weekly) than the
# monthly sheets, but they cover a LONGER history (from 1990) and at a
# higher frequency than our modelling grid. Rather than discard that extra
# information, or stretch it onto a daily grid full of stale repeats, we
# compress each calendar month's daily/weekly observations into summary
# statistics that describe the volatility and tail-risk *within* that
# month -- information a single month-end snapshot cannot capture.
daily = (pd.read_excel(xls, "Industry_D")
           .dropna(subset=["Date"])
           .sort_values("Date")
           .reset_index(drop=True))
weekly = (pd.read_excel(xls, "Industry_W")
            .dropna(subset=["Date"])
            .sort_values("Date")
            .reset_index(drop=True))
daily.columns = [c.strip() for c in daily.columns]
weekly.columns = [c.strip() for c in weekly.columns]
daily["ym"] = daily["Date"].dt.to_period("M")
weekly["ym"] = weekly["Date"].dt.to_period("M")

# Sign-robust simple return (handles the 20-Apr-2020 negative WTI print;
# see module docstring). Equivalent to a log return away from that event.
def robust_ret(s):
    return (s - s.shift(1)) / s.shift(1).abs()

daily["cl1_ret_d"] = robust_ret(daily["CL1"])
daily["brent_wti_spread_d"] = daily["Brent"] - daily["CL1"]
daily["d_brent_wti_spread_d"] = daily["brent_wti_spread_d"].diff()
daily["d_crack_d"] = daily["CRKS321C Index"].diff()

def _rms(x):
    """Root-sum-of-squares of daily changes = within-month realised vol."""
    x = x.dropna()
    return np.sqrt(np.nansum(x**2)) if len(x) else np.nan

def _max_drawdown(x):
    """Worst peak-to-trough cumulative move within the month."""
    x = x.dropna()
    if len(x) < 2:
        return np.nan
    cum = x.cumsum()
    return (cum - cum.cummax()).min()

daily_feats = daily.groupby("ym").agg(
    RV_CL1=("cl1_ret_d", _rms),                       # within-month realised vol of CL1
    DownDayFrac_CL1=("cl1_ret_d", lambda x: (x.dropna() < 0).mean() if x.dropna().shape[0] else np.nan),
    MaxDrawdown_CL1=("cl1_ret_d", _max_drawdown),      # worst within-month drawdown
    RV_BrentWTI=("d_brent_wti_spread_d", _rms),        # WTI-Brent dislocation volatility
    RV_CrackSpread=("d_crack_d", _rms),                # refining-margin volatility
).reset_index()

weekly["net_comm_w"] = weekly["Weekly Commercial Long"] - weekly["Weekly Commercial Short"]
weekly["d_net_comm_w"] = weekly["net_comm_w"].diff()

weekly_feats = weekly.groupby("ym").agg(
    Vol_WeeklyInvChg=("Weekly change in Crude Stock", lambda x: x.dropna().std()),
    Vol_WeeklyNetComm=("d_net_comm_w", lambda x: x.dropna().std()),
).reset_index()

df = df.merge(daily_feats, on="ym", how="left").merge(weekly_feats, on="ym", how="left")
print(f"Added {daily_feats.shape[1]-1} daily-derived + {weekly_feats.shape[1]-1} "
      f"weekly-derived within-month features (built from "
      f"{len(daily):,} daily rows and {len(weekly):,} weekly rows).")

# -----------------------------------------------------------------------
# 2. DROP BRENT LEVEL (redundant WTI benchmark; its *daily* dislocation
#    vs. CL1 is still captured above via RV_BrentWTI)
# -----------------------------------------------------------------------
df = df.drop(columns=["Brent"])

# -----------------------------------------------------------------------
# 3. TRANSFORMATIONS -> STATIONARY UNITS
# -----------------------------------------------------------------------
out = pd.DataFrame(index=df.index)
out["Date"] = df["Date"]

def log_ret(s):
    return np.log(s).diff()

def log_diff(s):
    return np.log(s).diff()

def simple_diff(s):
    return s.diff()

# --- Target & price-like variables (log returns) ---
out["ret_CL1"] = log_ret(df["CL1"])                       # <-- TARGET (y)
out["ret_CL2"] = log_ret(df["CL2"])
out["ret_Gasoline"] = log_ret(df["Gasoline All Grades"])
out["ret_DowJonesOilGas"] = log_ret(df["Dow Jones US Oil&Gas Index"])
out["ret_SPX"] = log_ret(df["SPX"])
out["ret_ExxonMobil"] = log_ret(df["ExxonMobil"])
out["ret_ConocoPhillips"] = log_ret(df["ConocoPhilips"])
out["ret_Chevron"] = log_ret(df["Chevron"])
out["ret_BP"] = log_ret(df["BP"])
out["ret_Shell"] = log_ret(df["Shell"])          # NaN pre Jul-2005 (ADR listing)
out["ret_TotalEnergies"] = log_ret(df["TotalEnergies"])

# futures curve shape: CL2-CL1 spread (contango/backwardation), in $ and change
out["spread_CL2_CL1"] = df["CL2"] - df["CL1"]
out["d_spread_CL2_CL1"] = out["spread_CL2_CL1"].diff()

# --- Quantity / level variables (log differences = growth rates) ---
out["d_Inventories"] = log_diff(df["Inventories"])
out["d_DailyProduction"] = log_diff(df["Daily Production"])
out["d_RigCount"] = log_diff(df["Rig Count"])
out["d_CommercialLong"] = log_diff(df["Commercial Long"])
out["d_CommercialShort"] = log_diff(df["Commercial Short"])
out["d_OpenInterest"] = log_diff(df["Total Open Interest"])
out["d_BalticDry"] = log_diff(df["Baltic Dry Index"])

# net commercial positioning (hedging pressure), % change
net_comm = df["Commercial Long"] - df["Commercial Short"]
out["d_NetCommercial"] = net_comm.pct_change()

# --- Spread-type variable (can be read as $ margin, use simple diff) ---
out["d_CrackSpread"] = simple_diff(df["CRKS321C Index"])

# --- Interest rates (first differences, i.e. change in yield) ---
out["d_3M_Yield"] = simple_diff(df["3M Yield"])
out["d_10Y_Yield"] = simple_diff(df["10Y Yield"])
out["d_TermSpread"] = simple_diff(df["10Y Yield"] - df["3M Yield"])

# --- Inflation & activity ---
out["CPI_YoY"] = df["CPI YOY %"]                 # already stationary-ish (bounded %)
out["d_CPI_YoY"] = df["CPI YOY %"].diff()         # change in inflation momentum
out["d_IndProd"] = log_diff(df["IndustrialProduction Index"])

# --- FX ---
out["ret_DXY"] = df["DXY returns"]                # already supplied as returns

# --- Volatility ---
out["VIX_level"] = df["VIX"]                      # kept in LEVEL for the regime rule
out["d_VIX"] = df["VIX"].diff()

# --- Within-month features engineered from the daily/weekly sheets (1b) ---
# These are already stationary by construction (RMS of daily/weekly changes,
# a bounded fraction, or a drawdown around zero), so they are used as-is,
# with no further log/diff transform.
out["RV_CL1"] = df["RV_CL1"]                      # within-month realised vol of CL1 (daily)
out["DownDayFrac_CL1"] = df["DownDayFrac_CL1"]    # fraction of down-days in the month
out["MaxDrawdown_CL1"] = df["MaxDrawdown_CL1"]    # worst within-month drawdown (<=0)
out["RV_BrentWTI"] = df["RV_BrentWTI"]            # WTI-Brent dislocation volatility (daily)
out["RV_CrackSpread"] = df["RV_CrackSpread"]      # refining-margin volatility (daily)
out["Vol_WeeklyInvChg"] = df["Vol_WeeklyInvChg"]  # within-month dispersion of weekly inventory chg
out["Vol_WeeklyNetComm"] = df["Vol_WeeklyNetComm"]# within-month dispersion of weekly net positioning

print("\nTransformed predictor panel shape:", out.shape)

# -----------------------------------------------------------------------
# 4. HIGH-VOLATILITY REGIME (EXPANDING, NO LOOK-AHEAD)
# -----------------------------------------------------------------------
# Expanding quantile uses ONLY information up to and including month t.
# We then LAG this flag along with the rest of the predictors in step 5,
# so the regime used to forecast month t's return is the regime observed
# at t-1.
vix = out["VIX_level"]
expanding_threshold = vix.expanding(min_periods=VIX_BURNIN).quantile(VIX_QUANTILE)
out["VIX_threshold"] = expanding_threshold
out["HighVol"] = (vix > expanding_threshold).astype(float)
out.loc[expanding_threshold.isna(), "HighVol"] = np.nan

print(f"High-vol regime flag built from expanding {int(VIX_QUANTILE*100)}th "
      f"percentile of VIX (burn-in = {VIX_BURNIN} months)")
print("Share of months flagged High-Vol (post burn-in):",
      round(out["HighVol"].mean(skipna=True), 3))

# -----------------------------------------------------------------------
# 5. LAG ALL PREDICTORS BY ONE MONTH (t-1 INFORMATION SET)
# -----------------------------------------------------------------------
target_col = "ret_CL1"
predictor_cols = [c for c in out.columns if c not in ("Date", target_col)]

lagged = out[predictor_cols].shift(1)
lagged.columns = [f"L1_{c}" for c in predictor_cols]

model_df = pd.concat([out[["Date", target_col]], lagged], axis=1)

# -----------------------------------------------------------------------
# 6. SEASONAL DUMMIES (keyed to the month being forecast -> NOT lagged)
# -----------------------------------------------------------------------
month_num = model_df["Date"].dt.month
month_dummies = pd.get_dummies(month_num, prefix="Month").astype(int)
month_dummies = month_dummies.drop(columns=["Month_1"])  # January = baseline

model_df["Summer"] = month_num.isin(SUMMER_MONTHS).astype(int)
model_df["Winter"] = month_num.isin(WINTER_MONTHS).astype(int)

model_df = pd.concat([model_df, month_dummies], axis=1)

# -----------------------------------------------------------------------
# 7. SEASONAL x HIGH-VOL INTERACTIONS
# -----------------------------------------------------------------------
model_df["Summer_x_HighVol"] = model_df["Summer"] * model_df["L1_HighVol"]
model_df["Winter_x_HighVol"] = model_df["Winter"] * model_df["L1_HighVol"]

# -----------------------------------------------------------------------
# 8. FINAL CLEAN-UP
# -----------------------------------------------------------------------

drop_cols = [c for c in model_df.columns
             if c.startswith("L1_ret_") and
             c.split("L1_ret_")[1] in
             ["ExxonMobil", "ConocoPhillips", "Chevron", "BP", "Shell", "TotalEnergies"]]
model_df = model_df.drop(columns=drop_cols)

core_needed = [c for c in model_df.columns if c not in ("Date",)]
before = len(model_df)
model_df_clean = model_df.dropna(subset=core_needed).reset_index(drop=True)
after = len(model_df_clean)

print(f"\nRows before burn-in/NaN drop: {before}")
print(f"Rows dropped (differencing lag=1 + {VIX_BURNIN}m VIX burn-in): {before - after}")
print(f"Final model-ready sample: {after} months, "
      f"{model_df_clean['Date'].min().date()} to {model_df_clean['Date'].max().date()}")
print(f"Final column count: {model_df_clean.shape[1]}")

# -----------------------------------------------------------------------
# 9. DATA DICTIONARY
# -----------------------------------------------------------------------
desc = {
    "Date": "Month-end date (the month whose CL1 return is being forecast)",
    "ret_CL1": "TARGET: log return of CL1 (front-month WTI futures) realised in month t",
}
for c in model_df_clean.columns:
    if c in desc or c == "Date":
        continue
    if c.startswith("L1_"):
        desc[c] = f"Predictor, lagged 1 month (known at t-1): {c[3:]}"
    elif c.startswith("Month_"):
        desc[c] = f"Calendar month dummy ({c}); baseline = January"
    elif c in ("Summer", "Winter"):
        desc[c] = f"Seasonal dummy for month t ({'May-Sep' if c=='Summer' else 'Nov-Mar'})"
    elif "x_HighVol" in c:
        desc[c] = "Interaction: seasonal dummy x lagged High-Vol regime flag"
    else:
        desc[c] = "See variable name"

data_dict = pd.DataFrame({"Variable": list(desc.keys()), "Description": list(desc.values())})

# -----------------------------------------------------------------------
# 10. EXPORT
# -----------------------------------------------------------------------
with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    model_df_clean.to_excel(writer, sheet_name="Model_Ready_Data", index=False)
    data_dict.to_excel(writer, sheet_name="Data_Dictionary", index=False)

model_df_clean.to_csv("model_ready_dataset.csv", index=False)

print(f"\nSaved: {OUTPUT_FILE} and model_ready_dataset.csv")

# -----------------------------------------------------------------------
# 11. DOWNLOAD (so you can cross-verify the cleaned data locally)
# -----------------------------------------------------------------------

try:
    from google.colab import files  # only exists inside Colab
    print("\nTriggering browser download of cleaned dataset...")
    files.download(OUTPUT_FILE)
    files.download("model_ready_dataset.xlsx")
except ImportError:
    print("\n(Not running in Google Colab -- skipping auto-download. "
          f"Files are saved in the working directory: {OUTPUT_FILE}, "
          "model_ready_dataset.csv)")


Using input file: ./Oil Data (2).xlsx
Aligned monthly panel: 355 months, 1994-12-31 to 2024-06-30
Added 5 daily-derived + 2 weekly-derived within-month features (built from 8,692 daily rows and 1,803 weekly rows).

Transformed predictor panel shape: (355, 39)
High-vol regime flag built from expanding 70th percentile of VIX (burn-in = 36 months)
Share of months flagged High-Vol (post burn-in): 0.297

Rows before burn-in/NaN drop: 355
Rows dropped (differencing lag=1 + 36m VIX burn-in): 36
Final model-ready sample: 319 months, 1997-12-31 to 2024-06-30
Final column count: 50

Saved: model_ready_dataset.xlsx and model_ready_dataset.csv

Triggering browser download of cleaned dataset...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>